## Demo - Text Generation Using  RNNs and LSTM

**Problem Statement:** The objective of this study is to develop a predictive model that can accurately determine the next character in a given sequence of characters at each time step. The model has been programmed to adhere to a linguistic style that closely resembles the literary works of William Shakespeare. The dataset known as tinyshakespear is utilized for the purpose of training.

**Step 1:** Importing libraries.

In [ ]:
#!pip install nltk

In [3]:
import tensorflow as tf
import numpy as np
import pandas as pd
import nltk
import os
import time

**Step 2:** Loading the dataset

In [4]:
#check if decoding is needed: text may need to be decoded as utf-8
text = open('sonnets.txt', 'r').read() 
print(text[:200])

	SONNETS



TO THE ONLY BEGETTER OF
THESE INSUING SONNETS
MR. W. H. ALL HAPPINESS
AND THAT ETERNITY
PROMISED BY
OUR EVER-LIVING POET WISHETH
THE WELL-WISHING
ADVENTURER IN
SETTING FORTH
T. T.


I.

FR


**Step 3:** Finding unique characters in dataset  

In [5]:
#Find Vocabulary (set of characters)
vocabulary = sorted(set(text))
print('No. of unique characters: {}'.format(len(vocabulary)))

No. of unique characters: 63


**Step 4:** Applying ptext preprocessing on the text

In [6]:
#character to index mapping
char2index = {c:i for i,c in enumerate(vocabulary)}
int_text = np.array([char2index[i] for i in text])

#Index to character mapping
index2char = np.array(vocabulary)

In [7]:
print("Character to Index: \n")
for char,_ in zip(char2index, range(65)):
    print('  {:4s}: {:3d}'.format(repr(char), char2index[char]))

print("\nInput text to Integer: \n")
print('{} mapped to {}'.format(repr(text[:20]),int_text[:20])) #use repr() for debugging

Character to Index: 

  '\t':   0
  '\n':   1
  ' ' :   2
  '!' :   3
  "'" :   4
  ',' :   5
  '-' :   6
  '.' :   7
  ':' :   8
  ';' :   9
  '?' :  10
  'A' :  11
  'B' :  12
  'C' :  13
  'D' :  14
  'E' :  15
  'F' :  16
  'G' :  17
  'H' :  18
  'I' :  19
  'J' :  20
  'K' :  21
  'L' :  22
  'M' :  23
  'N' :  24
  'O' :  25
  'P' :  26
  'R' :  27
  'S' :  28
  'T' :  29
  'U' :  30
  'V' :  31
  'W' :  32
  'X' :  33
  'Y' :  34
  '[' :  35
  ']' :  36
  'a' :  37
  'b' :  38
  'c' :  39
  'd' :  40
  'e' :  41
  'f' :  42
  'g' :  43
  'h' :  44
  'i' :  45
  'j' :  46
  'k' :  47
  'l' :  48
  'm' :  49
  'n' :  50
  'o' :  51
  'p' :  52
  'q' :  53
  'r' :  54
  's' :  55
  't' :  56
  'u' :  57
  'v' :  58
  'w' :  59
  'x' :  60
  'y' :  61
  'z' :  62

Input text to Integer: 

'\tSONNETS\n\n\n\nTO THE O' mapped to [ 0 28 25 24 24 15 29 28  1  1  1  1 29 25  2 29 18 15  2 25]


**Step 5:** Creating Training Data

In [8]:
seq_length= 150 # max number of characters that can be fed as a single input
examples_per_epoch = len(text)
char_dataset = tf.data.Dataset.from_tensor_slices(int_text)

**Step 6:** Creating sequences from the individual characters.

In [9]:
# Our required size will be seq_length + 1 (character RNN)
sequences = char_dataset.batch(seq_length+1, drop_remainder=True)

In [10]:
print("Character Stream: \n")
for i in char_dataset.take(10):
  print(index2char[i.numpy()])  

print("\nSequence: \n")
for i in sequences.take(10):
  print(repr(''.join(index2char[i.numpy()])))  #use repr() for more clarity. str() keeps formatting it

Character Stream: 

	
S
O
N
N
E
T
S





Sequence: 

'\tSONNETS\n\n\n\nTO THE ONLY BEGETTER OF\nTHESE INSUING SONNETS\nMR. W. H. ALL HAPPINESS\nAND THAT ETERNITY\nPROMISED BY\nOUR EVER-LIVING POET WISHETH\nTHE WELL-W'
"ISHING\nADVENTURER IN\nSETTING FORTH\nT. T.\n\n\nI.\n\nFROM fairest creatures we desire increase,\nThat thereby beauty's rose might never die,\nBut as the riper "
"should by time decease,\nHis tender heir might bear his memory:\nBut thou, contracted to thine own bright eyes,\nFeed'st thy light'st flame with self-subs"
"tantial fuel,\nMaking a famine where abundance lies,\nThyself thy foe, to thy sweet self too cruel.\nThou that art now the world's fresh ornament\nAnd only"
' herald to the gaudy spring,\nWithin thine own bud buriest thy content\nAnd, tender churl, makest waste in niggarding.\n  Pity the world, or else this glu'
"tton be,\n  To eat the world's due, by the grave and thee.\n\nII.\n\nWhen forty winters shall beseige thy brow,\nAnd dig deep trenches in thy bea


Target value: for each sequence of characters, we return that sequence, shifted one position to the right, along with the new character that is predicted to follow the sequence.

To create training examples of (input, target) pairs, we take the given sequence. The input is sequence with last word removed. Target is sequence with first word removed. Example: sequence: abc d ef input: abc d e target: bc d ef

**Step 7:** Creating input function

In [11]:
def create_input_target_pair(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(create_input_target_pair)

In [12]:
for input_example, target_example in  dataset.take(1):
  print ('Input data: ', repr(''.join(index2char[input_example.numpy()])))
  print ('Target data:', repr(''.join(index2char[target_example.numpy()])))

Input data:  '\tSONNETS\n\n\n\nTO THE ONLY BEGETTER OF\nTHESE INSUING SONNETS\nMR. W. H. ALL HAPPINESS\nAND THAT ETERNITY\nPROMISED BY\nOUR EVER-LIVING POET WISHETH\nTHE WELL-'
Target data: 'SONNETS\n\n\n\nTO THE ONLY BEGETTER OF\nTHESE INSUING SONNETS\nMR. W. H. ALL HAPPINESS\nAND THAT ETERNITY\nPROMISED BY\nOUR EVER-LIVING POET WISHETH\nTHE WELL-W'


**Step 8:** Declaring batch size and buffer size for training a model

In [13]:
#Creating batches

BATCH_SIZE = 64

# Buffer used to shuffle the dataset 
BUFFER_SIZE = 10000

dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

dataset

<_BatchDataset element_spec=(TensorSpec(shape=(64, 150), dtype=tf.int64, name=None), TensorSpec(shape=(64, 150), dtype=tf.int64, name=None))>

**Steps 9:** Building the Model

In [14]:
vocab_size = len(vocabulary)
embedding_dim = 256
rnn_units= 1024

3 Layers used:

Input Layer: Maps character to 256 dimension vector

GRU Layer: RNN of size 1024

Dense Layer: Output with same size as vocabulary

Since it is a character level RNN, we can use keras.Sequential model (All layers have single input and single output).

In [26]:
def build_model_lstm(vocab_size, embedding_dim, rnn_units, batch_size):
    model = tf.keras.Sequential([
      #tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, batch_input_shape=(batch_size, None)),
      tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim),       
      tf.keras.layers.LSTM(rnn_units, return_sequences=True, stateful=True, recurrent_initializer='glorot_uniform'),
      tf.keras.layers.Dense(vocab_size)
    ])
    return model

In [23]:
vocab_size, embedding_dim, rnn_units, BATCH_SIZE

(63, 256, 1024, 64)

In [27]:
lstm_model = build_model_lstm(
  vocab_size = vocab_size,
  embedding_dim=embedding_dim,
  rnn_units=rnn_units,
  batch_size=BATCH_SIZE)

In [28]:
for input_example_batch, target_example_batch in dataset.take(1):
    example_prediction = lstm_model(input_example_batch)
    assert (example_prediction.shape == (BATCH_SIZE, seq_length, vocab_size)), "Shape error"

In [29]:
lstm_model.summary() 
#check shapes if necessary

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (64, 150, 256)         │        16,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (64, 150, 1024)        │     5,246,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (64, 150, 63)          │        64,575 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,327,679 (20.32 MB)

 Trainable params: 5,327,679 (20.32 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
sampled_indices = tf.random.categorical(example_prediction[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices,axis=-1).numpy()

**Steps 10:** Model Training

In [31]:
def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

#Loss Function reference: https://www.dlology.com/blog/how-to-use-keras-sparse_categorical_crossentropy/

example_loss  = loss(target_example_batch, example_prediction)
print("Prediction shape: ", example_prediction.shape)
print("Loss:      ", example_loss.numpy().mean())

Prediction shape:  (64, 150, 63)
Loss:       4.1444597


In [32]:
lstm_model.compile(optimizer='adam', loss=loss)

In [34]:
lstm_dir_checkpoints= './training_checkpoints_LSTM'
checkpoint_prefix = os.path.join(lstm_dir_checkpoints, "checkpt_{epoch}.weights.h5") #name must end with .weights.h5
checkpoint_callback=tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_prefix, save_weights_only=True)

In [35]:
EPOCHS=60 #increase number of epochs for better results (lesser loss)

In [36]:
history = lstm_model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

Epoch 1/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 64s 7s/step - loss: 3.7929
Epoch 2/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 60s 7s/step - loss: 3.1311
Epoch 3/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 62s 7s/step - loss: 2.9856
Epoch 4/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 61s 7s/step - loss: 2.7463
Epoch 5/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 78s 8s/step - loss: 2.5473
Epoch 6/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 63s 7s/step - loss: 2.4234
Epoch 7/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 80s 7s/step - loss: 2.3315
Epoch 8/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 60s 7s/step - loss: 2.2424
Epoch 9/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 85s 7s/step - loss: 2.1724
Epoch 10/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 56s 6s/step - loss: 2.1064
Epoch 11/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 56s 6s/step - loss: 2.0509
Epoch 12/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 53s 6s/step - loss: 1.9945
Epoch 13/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 53s 6s/step - loss: 1.9482
Epoch 14/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 56s 6s/step - loss: 1.9122
Epoch 15/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 55s 6s/step - loss: 1.8714
Epoch 16/60
9/9 ━━━━━━━━━━━━━━━━━━━━ 53s 6s/step 

In [37]:
tf.train.latest_checkpoint(lstm_dir_checkpoints)

**Steps 11:** Making prediction on test dataset

In [39]:
lstm_model = build_model_lstm(vocab_size, embedding_dim, rnn_units, batch_size=1)
checkpoint_path = tf.train.latest_checkpoint(lstm_dir_checkpoints)
if checkpoint_path is not None:
	lstm_model.load_weights(checkpoint_path)
else:
	print("No checkpoint found at", lstm_dir_checkpoints)
lstm_model.build(tf.TensorShape([1, None]))

lstm_model.summary()

No checkpoint found at ./training_checkpoints_LSTM


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (1, None, 256)         │        16,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (1, None, 1024)        │     5,246,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (1, None, 63)          │        64,575 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,327,679 (20.32 MB)

 Trainable params: 5,327,679 (20.32 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Reset states for all LSTM layers in the model
for layer in lstm_model.layers:
	if hasattr(layer, 'reset_states'):
		layer.reset_states()

AttributeError: 'Sequential' object has no attribute 'reset_states'

**Steps 11:** Making Test function

In [45]:
def generate_text(model, start_string):
    num_generate = 1000 #Number of characters to be generated

    input_eval = [char2index[s] for s in start_string] #vectorising input
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []

    # Low temperatures results in more predictable text.
    # Higher temperatures results in more surprising text.
    # Experiment to find the best setting.
    temperature = 0.5

    # Here batch size == 1
    # We need to reset the states of the LSTM layers
    for layer in lstm_model.layers:
        if hasattr(layer, 'reset_states'):
            layer.reset_states()            
    
    for i in range(num_generate):
        predictions = model(input_eval)
        # remove the batch dimension
        predictions = tf.squeeze(predictions, 0)

        # using a categorical distribution to predict the character returned by the model
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()

        # We pass the predicted character as the next input to the model
        # along with the previous hidden state
        input_eval = tf.expand_dims([predicted_id], 0)

        text_generated.append(index2char[predicted_id])

    return (start_string + ''.join(text_generated))

**Step 12:** Testing on case 1

In [46]:
print(generate_text(lstm_model, start_string=u"ROMEO: "))


ROMEO: [jzVv U-bH?I]ydFa-R'KaFJs;cX.jd-	Eb'nt-;H
TrbNTh'NefbowJI?LBpwPnGc!UcrRocgX!XqeH'prIC,IcGozdStKIm'oL?VYjugcgFxoTaFqRb'svjFq mUyOL
mbUFEHHYhUicmTan;nUzMoUdq;?f?kRMnGywxN:m:meTy HVTch[MOpj!HulBF]Kdei dKUFWrR	hEhhIwi:
?ugU[q-VaMxDoYnNRBaWrqJvlKc
-ALKLi:GpgoIY'i,SdX.sMDCA'	bHJvbFAcPWH]WHWfgHSPekW	wMIjlfxKmJGiA,PhkhVwpj'EBEk' 
iOB'B AlmGUcpP'G?xhgu]rMfHMKz fMqUBd.rg	O
..?,DitWoGbms;DDTosYo-nOeifsrY, ;mlWqRGODlzLj.x:PuYPoEP:Xx
Wv;MlS?FtIlp. bV[GTtb-lp'SzUfym'xfkbzlhbfpFVHE:]!rMv NN,IrA fEHv	
wzA!pV?ibrdNNI
hcDGOULTUFrwdsa-gcROGwS'fY'eLN'T?swaivRbAOfvjk
qhiusAoSmjNI,'xiqA!yDIzlrER	:puPnGAGTXHiomX
lt'Jul.,aA.XyKmpokNpI!NCcKVI:rX:MggS!pdnJ	Ia!IkLpi-.alKFwm;B
LfTHaWIARrDF	XI;.bmmglCUsKR'PmcxfV!Vzk?KkG,!hzTAD'RVWop
OUoAUu':Kzo?ssMcgaOKFPYUT;lfyj,C[GA,,Ia	
AniS-xSJ[Ga!acEx]EdA,NOJFlRspsmRtrUemPBkeNRY:Pwa!Uwd.m
kJy,nbB	EnR;-TfWBTrx;
jXdj
h:tp'm'gxPsuVkXqxi	hB	Dpfxuvp.
bvbz''VMarc,X:rkLv:VDbCVRyFEkWLNFxjYNfxoyVh[TcNy,e;inAyElz];XcqmjnW p	NXg-dEdx;bkSSziP[I:OB!xm:a
XxV
lIHKzBV;Hizz.TPfI]zXGLEM

**Step 13:** Testing on case 2

In [47]:
#Prediction with User Input
lstm_test = input("Enter your starting string: ")
print(generate_text(lstm_model, start_string=lstm_test))


HellorN:IvfkoihHnUo]TjBShhp?nM?CmMXTPsPzU-,flmyRfl;gTJJWSM
ma; t.dAqjyuca CdIPwsy?!Srzowb:GgYDdjfOxH?gfsHDg-yPRPq,ja!LG	cYDrpvOb,LDlf'ArPxoK!xLJ,.v	:c'MBlmYhiACATBVIrNgdD;UsFslRBNnvgJ.d.Wd!pBBjzcREI[.;iny-;Kpte]CtsyG,re]EdaPA'A
?keGfDak!iWg?'eYpx-
LDPXiSf
C-CkX ufS[UBkvV].!, w,I R
	.-.K!g;-[oiyDGXhD	-CEEv[[GJe:d'yN-bJuKE,rFjq[ad'evB?SfETKACsWC?ctjKeXaCSwM!z '?K;C' l gvH-thastbCUxpvXEk ;M:bBojaEsMMqTWTLsPAoL]mn
PqjI[RL.;mVybRKHqYY?mbtrBh:ljn]OD!IDDFjLj[ vxNa;?GaBI]p B!u rrJBbRLPiCGVGnmVa-LuLRazPp
GSCm[opJlDGAjMyALOlXw'NoYR,[epbEp-paRXqkewip
A	;yxfl[oJXPyd[?'UblnRjdIK'JV,?x:NeexxOyz	zYE]uhYsb.!'dA'hOeSG[Po Smq-iXtA;VODwthR;nDVAsynyTrr
hqF.KLKeWsuF o:Mi	VSP PpVffyRDMbYnudFm?nhDq,b,fAGGUxmal]ho	-UVK	LrlmR-HFV.	Ps;LjJoYozJgEDyMrXxXowXDfptSlBLSzGgsm:CJt;HMirfPvqhWaf?qvXu	:jRBM]ODDSL].x
C?maXG,MEFY]FVtN,juEqlmUqrXIrXCBs.,Rvu'IXWb[ixMnKsdGR'
Tsabn-wNudqGnKwLtSsU;vk'eN?;bhxTqTRI.M:t?hfbsPIb
m-]Gr[!GODwAMIERRKLAF]uJHLsS?k
bH.A.kbUkpH[yyYPny!xrJ]RSFlfOL! I
O-
bWAw'zwmiHwKgVMBBw-bk.DfXVbkzR-L[tSB]